# Viveka — GRPO Training on Kaggle

Train Qwen2-0.5B-Instruct with TRL GRPO + Unsloth 4-bit QLoRA on the Viveka OpenEnv.

## Prereqs (do these BEFORE running)

1. Notebook **Settings**: Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`
2. **Add-ons → Secrets**: add `GH_TOKEN` (GitHub PAT, repo write access) and `HF_TOKEN` (HuggingFace read token)
3. Run cells **in order**. Don't run Cell 6 (full training) until Cell 5 (smoke) finishes cleanly.

Total wall time: ~5–15 min smoke, ~2–6 hours full run, +5 min plots/push.

In [ ]:
# ── Cell 1: GPU check + clone + tokens ──────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

# Move to a known-good parent BEFORE any rm/clone — if the kernel was
# previously inside /kaggle/working/viveka-env and we rm -rf it, the
# shell loses its CWD and every subsequent command fails with
# 'getcwd: cannot access parent directories'.
os.chdir("/")
os.chdir("/kaggle/working")
%cd /kaggle/working

!nvidia-smi

# Clone fresh
GH_TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/viveka-env
!git clone https://{GH_TOKEN}@github.com/gowtham-sai-yadav/viveka-env.git /kaggle/working/viveka-env
%cd /kaggle/working/viveka-env

# Set HF_TOKEN for fast model download
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("\nTokens set:", bool(os.environ.get("HF_TOKEN")))

!git log --oneline -5

## Cell 2: Install — pin every layer that's bitten us

Order matters. Each pin solves a specific known issue:
- `openenv-core==0.2.2` (0.2.3 wheel has stale fastmcp imports)
- `fastmcp==3.1.1` (3.2 removed `CallToolResult`, 2.x is wrong era)
- `mcp` upgrade (Kaggle's preinstalled is too old, missing `Icon` class)
- `trl>=0.13` (GRPOConfig added in 0.13)
- `mergekit` (TRL transitive dep)
- `unsloth` (4-bit QLoRA training)
- `bitsandbytes` (4-bit quantisation backend)

In [ ]:
# ── Cell 2: Install all deps in known-good order ────────────────────

# Step 1: project itself in editable mode (uses pyproject pins)
!pip install -q -e ".[train]"

# Step 2: force-pin the openenv-core / fastmcp pair that works
!pip install --upgrade --force-reinstall --no-deps "openenv-core==0.2.2" "fastmcp==3.1.1"

# Step 3: upgrade mcp (Kaggle's preinstalled mcp lacks Icon class — fastmcp 3.1.1 imports it)
!pip install -q -U "mcp"

# Step 4: TRL >= 0.13 for GRPOConfig (Kaggle's preinstalled trl 1.2 is fine)
!pip install -q -U "trl>=0.13.0"

# Step 5: mergekit (TRL transitive)
!pip install -q mergekit

# Step 6: Unsloth (Kaggle/Colab compatible)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Step 7: fresh bitsandbytes for 4-bit quant
!pip install -q -U bitsandbytes

print("\n=== installed versions ===")
!pip show openenv-core fastmcp mcp trl transformers huggingface-hub peft 2>&1 | grep -E "^(Name|Version)"

In [ ]:
# ── Cell 3: Verify all critical imports clean ───────────────────────
import importlib, sys

def test(module):
    try:
        importlib.import_module(module)
        print(f"\u2705 {module}")
        return True
    except Exception as e:
        print(f"\u274c {module}: {type(e).__name__}: {e}")
        return False

ok = True
ok &= test("trl")
ok &= test("openenv.core.rubrics.trajectory")
ok &= test("openenv.core.env_server.types")
ok &= test("openenv.core.env_server.http_server")

try:
    from trl import GRPOConfig, GRPOTrainer
    print("\u2705 trl.GRPOConfig + GRPOTrainer")
except Exception as e:
    print(f"\u274c GRPO: {e}"); ok = False

try:
    from unsloth import FastLanguageModel
    print("\u2705 unsloth.FastLanguageModel")
except Exception as e:
    print(f"\u274c unsloth: {e}"); ok = False

try:
    sys.path.insert(0, "/kaggle/working/viveka-env")
    from viveka.server.environment import VivekaEnvironment
    print("\u2705 viveka.server.environment.VivekaEnvironment")
except Exception as e:
    print(f"\u274c viveka env: {e}"); ok = False

print(f"\n{'\u2705 ALL CLEAN — proceed' if ok else '\u274c FIX BEFORE PROCEEDING'}")

In [ ]:
# ── Cell 4: Dry-run (no GPU touch, ~10 sec) ─────────────────────────
!python train.py --dry-run

## Smoke run (10 episodes, ~10–20 min on T4)

Watch for:
1. `[load] Qwen/Qwen2-0.5B-Instruct via Unsloth 4-bit ...`
2. Training step logs with `loss` and `grad_norm`
3. **No `[NaNGuard]` halts**
4. Final `[smoke] terminal reward=...` line

If smoke crashes — STOP, paste the error, do NOT run Cell 6.

In [ ]:
# ── Cell 5: Smoke run (10 episodes) ─────────────────────────────────
!python train.py --smoke --output-dir /kaggle/working/runs/smoke --no-wandb

In [ ]:
# ── Cell 6: Full training (200 episodes, ~2–6 hours on T4) ──────────
# DO NOT RUN until Cell 5 (smoke) finishes cleanly.
# Sample 5 generations every 30 min while this runs — look for reward hacking.

!python train.py \
    --episodes 200 \
    --output-dir /kaggle/working/runs/v1 \
    --tier-mix "1:0.4,2:0.4,4:0.2" \
    --no-wandb

In [ ]:
# ── Cell 7: Generate the 3 hero plots from training logs ────────────
import os
os.makedirs("/kaggle/working/runs/v1", exist_ok=True)

# Reward curve (from training_log.jsonl)
!python eval/reward_curve.py /kaggle/working/runs/v1/training_log.jsonl \
    --output /kaggle/working/runs/v1/reward_curve.png 2>&1 | tail -5

# Reliability diagram (random + trained on same axes)
!python eval/reliability_diagram.py \
    --policies trained,random \
    --output /kaggle/working/runs/v1/reliability.png 2>&1 | tail -5

# AQI probe on trained model
!python eval/aqi_probe.py \
    --model /kaggle/working/runs/v1/lora \
    --output /kaggle/working/runs/v1/aqi_trained.json 2>&1 | tail -5

!ls -la /kaggle/working/runs/v1/

In [ ]:
# ── Cell 8: Push results back to GitHub ─────────────────────────────
!mkdir -p output
!cp /kaggle/working/runs/v1/reward_curve.png output/ 2>/dev/null && echo 'reward_curve copied' || echo 'no reward_curve'
!cp /kaggle/working/runs/v1/reliability.png output/ 2>/dev/null && echo 'reliability copied' || echo 'no reliability'
!cp /kaggle/working/runs/v1/aqi_trained.json output/ 2>/dev/null && echo 'aqi copied' || echo 'no aqi'
!cp /kaggle/working/runs/v1/training_log.jsonl output/ 2>/dev/null && echo 'log copied' || echo 'no log'

!git config user.email "gowtham.g@scaler.com"
!git config user.name "gowtham-sai-yadav"
!git add output/
!git status --short
!git commit -m "feat(training): GRPO run v1 — real reward curve + reliability + AQI from Kaggle"
!git push origin main

print("\n\u2705 results pushed to GitHub. Tell main session.")